## 1. Preparación

Conecto Google Drive para cargar el dataset unificado (1.400 registros,
200 por categoría) que armamos en los pasos anteriores, y voy a usarlo para
entrenar el primer modelo de clasificación del proyecto.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

RUTA_DATASET = '/content/drive/MyDrive/Datasets_TechMind/1 Semana - con el Dataset final con todo incluido - Repo GitHub/procesados/dataset_FINAL_UNIFICADO_techmind.csv'
df = pd.read_csv(RUTA_DATASET)

print("Filas totales:", len(df))
print(df["categoria"].value_counts())

Mounted at /content/drive
Filas totales: 1400
categoria
Backend           200
Bases de Datos    200
Cloud             200
Data Science      200
DevOps            200
Frontend          200
Mobile            200
Name: count, dtype: int64


## 2. Limpieza de texto

Antes de que el modelo pueda aprender de los textos, hay que normalizarlos:
enbtoncces aqui pasamos todo a minúsculas, sacamos signos de puntuación, y eliminamos las palabras que no aportan significado (como "el", "la", "de" — llamadas "stopwords").
Esta cosa evita que el modelo confunda variaciones de escritura con conceptos distintos. Esta misma función la va a necesitar Backend más adelante, para limpiar el texto nuevo que llegue por la API de la misma forma en que se limpió el texto de entrenamiento.

In [2]:
import nltk
nltk.download('stopwords')

import re
from nltk.corpus import stopwords

stopwords_espanol = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-záéíóúñ\s]', ' ', texto)  # saca números, símbolos y puntuación
    palabras = texto.split()
    palabras_sin_stopwords = [p for p in palabras if p not in stopwords_espanol]
    return ' '.join(palabras_sin_stopwords)

df["texto_limpio"] = df["texto"].apply(limpiar_texto)

print("Ejemplo antes:")
print(df["texto"].iloc[0][:200])
print("\nEjemplo después:")
print(df["texto_limpio"].iloc[0][:200])

Ejemplo antes:
El sistema de cerraduras de puertas inteligentes basado en el concepto de Internet de las cosas con backend móvil como servicio es el desarrollo de cerraduras de puertas inteligentes respaldado por la

Ejemplo después:
sistema cerraduras puertas inteligentes basado concepto internet cosas backend móvil servicio desarrollo cerraduras puertas inteligentes respaldado tecnología computación nube almacenamiento datos mét


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 3. Vectorización TF-IDF

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer(max_features=3000)
X = vectorizador.fit_transform(df["texto_limpio"])

print("Forma de la matriz (filas, columnas):", X.shape)
print("Ejemplo de palabras en el vocabulario:", vectorizador.get_feature_names_out()[:10])

Forma de la matriz (filas, columnas): (1400, 3000)
Ejemplo de palabras en el vocabulario: ['abajo' 'abarca' 'abc' 'abierto' 'aborda' 'abordar' 'abra' 'absoluta'
 'absoluto' 'abstracción']


## 4. Entrenar el modelo baseline

Aqui separe el dataset en dos partes: 80% para que el modelo aprenda (entrenamiento) y 20% para probarlo después con textos que nunca vio (prueba) — así sabemos si realmente aprendió el patrón, y no solo memorizó los ejemplos. Entreno con Regresión Logística, un algoritmo simple.
Esto es como primera versión antes de optimizar nada. Aqui podemos aportar todos. :)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, df["categoria"], test_size=0.2, random_state=42
)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_entrenamiento, y_entrenamiento)

print("✅ Modelo entrenado")
print("Ejemplos de entrenamiento:", X_entrenamiento.shape[0])
print("Ejemplos de prueba:", X_prueba.shape[0])

✅ Modelo entrenado
Ejemplos de entrenamiento: 1120
Ejemplos de prueba: 280


## 5. Evaluar el modelo

Aqui use el 20% de datos que el modelo nunca vio para medir qué tan bien predice cada categoría. El classification_report muestra, por categoría, qué porcentaje de aciertos tuvo (métrica F1) — así identificamos si alguna categoría quedó floja y necesita más ejemplos o ajustes más adelante.

In [5]:
from sklearn.metrics import classification_report

predicciones = modelo.predict(X_prueba)

print(classification_report(y_prueba, predicciones))

                precision    recall  f1-score   support

       Backend       0.56      0.63      0.59        35
Bases de Datos       0.71      0.71      0.71        42
         Cloud       0.72      0.67      0.69        42
  Data Science       0.75      0.88      0.81        43
        DevOps       0.81      0.74      0.77        46
      Frontend       0.81      0.62      0.70        40
        Mobile       0.61      0.69      0.65        32

      accuracy                           0.71       280
     macro avg       0.71      0.71      0.70       280
  weighted avg       0.72      0.71      0.71       280



## 6. Guardar el modelo

Aqui guardamos el modelo entrenado y el vectorizador TF-IDF en dos archivos, para que backend pueda cargarlos con joblib.load() e integrarlos a la API. ambos archivos van juntos: el vectorizador es necesario para convertir texto nuevo al mismo formato numérico con el que se entrenó el modelo.

In [8]:
import joblib
import os

CARPETA_MODELOS = '/content/drive/MyDrive/Datasets_TechMind/modelo baseline'
os.makedirs(CARPETA_MODELOS, exist_ok=True)

joblib.dump(modelo, f'{CARPETA_MODELOS}/modelo.pkl')
joblib.dump(vectorizador, f'{CARPETA_MODELOS}/vectorizer.pkl')

print("✅ Modelo y vectorizador guardados en:", CARPETA_MODELOS)

✅ Modelo y vectorizador guardados en: /content/drive/MyDrive/Datasets_TechMind/modelo baseline


## 7. Prueba con un texto nuevo

prueba con un texto que nunca nuevo —
La idea de aqui es simyular lo que va a pasar cuando llegue contenido real por la API.

In [7]:
texto_de_prueba = """
Docker es una plataforma que permite empaquetar una aplicación junto con
todas sus dependencias en un contenedor, para que funcione igual sin
importar en qué máquina se ejecute. Kubernetes se usa para orquestar
muchos contenedores Docker en producción.
"""

texto_prueba_limpio = limpiar_texto(texto_de_prueba)
texto_prueba_vectorizado = vectorizador.transform([texto_prueba_limpio])

prediccion = modelo.predict(texto_prueba_vectorizado)[0]
probabilidades = modelo.predict_proba(texto_prueba_vectorizado)[0]
probabilidad_maxima = max(probabilidades)

print("Categoría predicha:", prediccion)
print("Probabilidad:", round(probabilidad_maxima, 2))

Categoría predicha: Cloud
Probabilidad: 0.31
